SETUP

In [1]:
import pandas as pd
import sqlite3
import openpyxl

In [2]:
mer_df = pd.read_csv("csv_file/merchants.csv")
refund_df = pd.read_csv("csv_file/refunds.csv")
tran_info_df = pd.read_csv("csv_file/transaction_info.csv")
tran_df = pd.read_csv("csv_file/transactions.csv")

In [3]:
display(mer_df[:1])
display(refund_df[:1])
display(tran_info_df[:1])
display(tran_df[:1])

,merchant_id,leader_name,create_date,address,country,email,phone,industry
0,Richards-Little,Ms. Rachel Baker,2025-10-23,"7251 Patterson Station, New Jeremyshire, IL 72094",China,maciasandrew@simpson.info,206-857-7069x1009,Food & Beverage


,refund_id,tran_id,channel,amount,create_date,refund_date,reason,status
0,d0c37a23-9663-4410-89d3-62cfe87bb5e0,4a8c5a13-45de-451d-98ba-711f2cd9650a,DuitNow QR,45.74,2025-06-18 10:55:17,2025-06-25 10:55:17,Customer Dissatisfaction,completed


,tran_id,country,card_type,card_brands,region,channel_type
0,377a06dd-16de-4064-a464-da56c85039c8,Malaysia,NaN,NaN,Domestic,Buy Now Pay Later


,tran_id,merchant_id,channel,amount,currency,date,status
0,377a06dd-16de-4064-a464-da56c85039c8,"Townsend, Bishop and Mccoy",Shopee PayLater,46.95,MYR,2025-12-05 11:31:17,completed


In [4]:
# Create an in-memory SQLite database
conn = sqlite3.connect(":memory:")

# Load DataFrames into SQL tables
mer_df.to_sql("merchants", conn, index=False, if_exists='replace')
refund_df.to_sql("refunds", conn, index=False, if_exists='replace')
tran_info_df.to_sql("transaction_info", conn, index=False, if_exists='replace')
tran_df.to_sql("transactions", conn, index=False, if_exists='replace')

545701

GLOBAL FUNCTION

In [18]:
def convert_currency_to_MYR(dataframe):
    rates = {
        "MYR": 1,
        "SGD": 0.3042,
        "THB": 7.5380,
        "USD": 0.2372,
        "GBP": 0.1753,
        "CNY": 1.6897
    }

    dataframe = dataframe.copy()

    dataframe["amount_MYR"] = round(dataframe.apply(
        lambda x: x["amount"] / rates.get(x["currency"], 1), axis=1
    ), 2)
    
    return dataframe

In [19]:
def apply_refund_rate_lvl(rate):
    if rate > 6:
        return 'high'
    elif 5 < rate <= 6:
        return 'average'
    else:
        return 'low'

QUERY

In [20]:
sqlInput = """
SELECT merchant_id, country, email, phone
FROM merchants
WHERE country IN ('Malaysia', 'China', 'Singapore', 'Thailand')
"""

asiaMer = pd.read_sql_query(sqlInput, conn)

display(asiaMer)

asiaMer_list = tuple(asiaMer['merchant_id'].tolist())
print(len(asiaMer_list))

,merchant_id,country,email,phone
0,Richards-Little,China,maciasandrew@simpson.info,206-857-7069x1009
1,"Carlson, Lyons and Gonzalez",China,dereksaunders@rogers.com,(219)491-7442x3651
2,"Roman, Willis and Rojas",Singapore,sewing@frazier.com,+1-720-656-7585
3,"Colon, Jensen and Russell",Malaysia,terri58@washington-young.com,965-993-2903
4,Chapman and Sons,Thailand,shawnharper@castro.com,+1-217-573-3807
...,...,...,...,...
64,Williams-Walker,Thailand,grahamerin@medina.net,(900)419-3899
65,Morrow-Moore,China,jenniferknox@collins.biz,9483251049
66,"Cowan, Ritter and Nichols",Malaysia,zhines@martinez-conner.com,323.708.1943x966
67,Evans-Flynn,Thailand,landrytoni@sutton-jones.com,001-773-919-9689


69


In [21]:
sqlInput = f"""
SELECT t.tran_id, t.status, t.date, t.merchant_id, t.channel, t.amount, t.currency, 
    ti.card_type, ti.channel_type, 
    COUNT(DISTINCT r.tran_id) AS refund_count, SUM(r.amount) AS refund_amt, r.reason
FROM transactions t
    LEFT JOIN transaction_info ti on t.tran_id = ti.tran_id
    LEFT JOIN refunds r on t.tran_id = r.tran_id AND r.status = 'completed'
WHERE t.merchant_id IN {asiaMer_list}
    AND t.status IN ('completed', 'recorded', 'cancelled')
    AND t.date BETWEEN '2024-01-01 00:00:00' AND '2024-12-31 23:59:59'
GROUP BY t.tran_id
"""

asiaTran = pd.read_sql_query(sqlInput, conn)

display(asiaTran)

,tran_id,status,date,merchant_id,channel,amount,currency,card_type,channel_type,refund_count,refund_amt,reason
0,0000e95f-7123-4760-ac20-aea85f97c82e,completed,2024-06-26 09:55:30,Mckinney-Cruz,DuitNow QR,26.36,MYR,None,E-wallet,0,NaN,None
1,00015a81-2a7c-443b-8e47-6ef38005a085,completed,2024-03-28 09:39:53,Chapman and Sons,DuitNow QR Offline,6.24,MYR,None,E-wallet Offline,0,NaN,None
2,00016ebe-f545-44f2-af5f-d3c617c4ccd3,completed,2024-06-15 13:09:38,Warren-Avila,Shopee PayLater,65.58,MYR,None,Buy Now Pay Later,0,NaN,None
3,000188e0-6aa5-48f7-b22b-f1fb61400421,completed,2024-04-22 11:58:13,"Valdez, Moon and Miller",DuitNow QR,5.96,THB,None,E-wallet,0,NaN,None
4,0001bbeb-5329-4da4-ac6c-45198d5914a8,completed,2024-10-15 13:00:03,Richardson-Goodman,FPX,27.71,MYR,None,Online Bank,0,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...
109151,fffc711c-7cb8-4e2b-8afd-6e9d95656686,completed,2024-11-22 13:55:16,Mckinney-Cruz,TNG eWallet,10.26,MYR,None,E-wallet,0,NaN,None
109152,fffd4641-861c-4bea-bd0d-ad6bdd35110d,completed,2024-04-12 10:07:02,Ibarra Ltd,Mastercard,42.62,MYR,debit,Credit Card Offline,1,42.62,Other
109153,fffe02fa-319b-42e4-8ec9-05e8a1547550,cancelled,2024-01-04 17:40:34,Cruz-Henry,TNG eWallet,50.62,MYR,None,E-wallet,1,50.62,Other
109154,fffee48a-0b06-4fc7-85be-32f325d8ebe9,completed,2024-08-06 22:34:32,Carter Group,Apple Pay,36.48,MYR,debit,Credit Card Online,0,NaN,None


In [22]:
asiaTran['refund_amt'] = asiaTran['refund_amt'].fillna(0)
converted_asiaTran = convert_currency_to_MYR(asiaTran)
converted_asiaTran = converted_asiaTran.rename(columns={'amount':'txn_amt', 'amount_MYR':'txn_amt_MYR', 'refund_amt':'amount'})
converted_asiaTran = convert_currency_to_MYR(converted_asiaTran)
converted_asiaTran = converted_asiaTran.rename(columns={'amount_MYR':'refund_amt_MYR'})
display(converted_asiaTran[converted_asiaTran['refund_amt_MYR'] != 0][:5])

,tran_id,status,date,merchant_id,channel,txn_amt,currency,card_type,channel_type,refund_count,amount,reason,txn_amt_MYR,refund_amt_MYR
30,000ebdd7-ade7-4677-9c54-82ebd31ab87a,completed,2024-08-10 13:36:15,Williams-Walker,Visa,142.41,USD,credit,Credit Card Offline,1,142.41,Customer Dissatisfaction,600.38,600.38
33,001092ea-18fc-45e7-bf59-1194d41e6fbb,completed,2024-12-31 11:52:44,Warren-Avila,DuitNow QR,91.98,SGD,None,E-wallet,1,91.98,Wrong Item Sent,302.37,302.37
41,001abe6f-b4ef-4b91-a995-3878b0dcca93,cancelled,2024-01-14 20:25:02,Williams-Walker,Apple Pay,41.23,CNY,debit,Credit Card Online,1,41.23,Other,24.40,24.40
87,0037bc37-fb80-4b74-92f2-6cbc8158bfa9,completed,2024-08-13 10:07:32,Barnes-Taylor,TNG eWallet Offline,30.95,CNY,None,E-wallet Offline,1,30.95,Wrong Item Sent,18.32,18.32
89,003b3d36-19f9-4cf9-8472-936847fde6cf,cancelled,2024-05-02 16:34:30,Warren-Avila,TNG eWallet,65.33,MYR,None,E-wallet,1,65.33,Fraudulent Transaction,65.33,65.33


In [23]:
converted_asiaTran["partial_refund_count"] = converted_asiaTran.apply(
    lambda x: 1 if (x["txn_amt_MYR"] > 0 and 0 < x["refund_amt_MYR"] < x["txn_amt_MYR"]) else 0,
    axis=1
)

converted_asiaTran["fully_refund_count"] = converted_asiaTran.apply(
    lambda x: 1 if (x["txn_amt_MYR"] > 0 and x["refund_amt_MYR"] == x["txn_amt_MYR"]) else 0,
    axis=1
)

grouped_asiaTran = converted_asiaTran.groupby(["merchant_id"]).agg(
    channel=('channel', lambda x: ', '.join(map(str, x.unique()))),
    txn_count=('status', lambda x: x.isin(['completed', 'recorded']).sum()),
    refund_count=('refund_count', 'sum'),
    txn_amt=('txn_amt_MYR', 'sum'),
    refund_amt=('refund_amt_MYR', 'sum'),
    partial_refund_count=('partial_refund_count', 'sum'),
    fully_refund_count=('fully_refund_count', 'sum')
).reset_index()

display(grouped_asiaTran[:5])

,merchant_id,channel,txn_count,refund_count,txn_amt,refund_amt,partial_refund_count,fully_refund_count
0,Acosta PLC,"TNG eWallet, TNG eWallet Offline, Mastercard, ...",594,42,60109.20,3810.29,3,39
1,Aguilar PLC,"FPX, DuitNow QR Offline, Google Pay, DuitNow Q...",1796,129,181081.10,14268.39,21,108
2,"Aguilar, Roberts and Pruitt","DuitNow QR Offline, FPX, DuitNow QR, TNG eWall...",1785,130,163808.10,12010.41,19,111
3,Barnes-Taylor,"TNG eWallet Offline, DuitNow QR, Visa, DuitNow...",1656,104,159995.95,8096.41,16,88
4,Benton LLC,"TNG eWallet, TNG eWallet Offline, FPX, DuitNow...",1789,104,169058.93,8769.98,17,87


In [24]:
grouped_asiaTran['refund_rate'] = round((grouped_asiaTran['refund_count']/grouped_asiaTran['txn_count'])*100, 2)
grouped_asiaTran['partial_refund_rate'] = round((grouped_asiaTran['partial_refund_count']/grouped_asiaTran['refund_count'])*100, 2)
grouped_asiaTran['fully_refund_rate'] = round((grouped_asiaTran['fully_refund_count']/grouped_asiaTran['refund_count'])*100, 2)
grouped_asiaTran['refund_lvl'] = grouped_asiaTran['refund_rate'].apply(apply_refund_rate_lvl)
display(grouped_asiaTran[:5])

,merchant_id,channel,txn_count,refund_count,txn_amt,refund_amt,partial_refund_count,fully_refund_count,refund_rate,partial_refund_rate,fully_refund_rate,refund_lvl
0,Acosta PLC,"TNG eWallet, TNG eWallet Offline, Mastercard, ...",594,42,60109.20,3810.29,3,39,7.07,7.14,92.86,high
1,Aguilar PLC,"FPX, DuitNow QR Offline, Google Pay, DuitNow Q...",1796,129,181081.10,14268.39,21,108,7.18,16.28,83.72,high
2,"Aguilar, Roberts and Pruitt","DuitNow QR Offline, FPX, DuitNow QR, TNG eWall...",1785,130,163808.10,12010.41,19,111,7.28,14.62,85.38,high
3,Barnes-Taylor,"TNG eWallet Offline, DuitNow QR, Visa, DuitNow...",1656,104,159995.95,8096.41,16,88,6.28,15.38,84.62,high
4,Benton LLC,"TNG eWallet, TNG eWallet Offline, FPX, DuitNow...",1789,104,169058.93,8769.98,17,87,5.81,16.35,83.65,average


In [25]:
asia_merged = pd.merge(asiaMer, grouped_asiaTran, on='merchant_id', how='left')
asia_merged = asia_merged.sort_values(by=['refund_rate'], ascending=[False] )
display(asia_merged)

,merchant_id,country,email,phone,channel,txn_count,refund_count,txn_amt,refund_amt,partial_refund_count,fully_refund_count,refund_rate,partial_refund_rate,fully_refund_rate,refund_lvl
12,Cox-Strong,Thailand,kathrynrobinson@gallegos.org,680-374-5677x459,"TNG eWallet, DuitNow QR Offline, FPX, Masterca...",1139,96,120153.21,6068.22,16,80,8.43,16.67,83.33,high
3,"Colon, Jensen and Russell",Malaysia,terri58@washington-young.com,965-993-2903,"FPX, Mastercard, Google Pay, DuitNow QR, TNG e...",1396,112,139420.26,18540.53,19,93,8.02,16.96,83.04,high
24,Dixon-Johnson,Singapore,uhenry@edwards.com,997-312-2216x19636,"DuitNow QR Offline, DuitNow QR, TNG eWallet, F...",503,39,47813.01,1458.71,3,36,7.75,7.69,92.31,high
30,Phillips PLC,Malaysia,santosstephanie@bell-weber.com,+1-577-306-4096x6367,"DuitNow QR, Visa, Apple Pay, TNG eWallet Offli...",740,57,71630.14,2254.24,10,47,7.70,17.54,82.46,high
32,Logan Inc,Malaysia,toddjoshua@wright.com,901-743-6865,"TNG eWallet, DuitNow QR, TNG eWallet Offline, ...",641,49,58643.17,5696.82,10,39,7.64,20.41,79.59,high
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44,Benton LLC,Malaysia,chris33@smith.com,001-571-479-4551,"TNG eWallet, TNG eWallet Offline, FPX, DuitNow...",1789,104,169058.93,8769.98,17,87,5.81,16.35,83.65,average
54,"Marshall, Paul and Nelson",China,kimberlycoleman@morgan.net,4753447107,"FPX, DuitNow QR Offline, Mastercard, DuitNow Q...",1158,67,100806.22,5215.53,10,57,5.79,14.93,85.07,average
33,Ellison-Park,Malaysia,lewiserica@miller-hunter.com,001-958-395-5703,"DuitNow QR, TNG eWallet, FPX, Visa, DuitNow QR...",1355,72,127207.41,6881.11,12,60,5.31,16.67,83.33,average
35,Lee-White,Malaysia,jenniferjones@roman.com,+1-680-946-2071x324,"DuitNow QR, Mastercard, TNG eWallet, Visa, Dui...",1670,86,160879.49,6865.08,14,72,5.15,16.28,83.72,average


In [26]:
asia_merged = asia_merged.drop(columns={'partial_refund_count', 'fully_refund_count'})
asia_merged['txn_amt'] = round(asia_merged['txn_amt'], 2)
asia_merged['refund_amt'] = round(asia_merged['refund_amt'], 2)
asia_merged = asia_merged.rename(columns={'merchant_id':'Merchant ID', 'country':'Country', 'email':'Email', 'phone':'Phone Number',
                                         'channel':'Channel', 'txn_count':'Transaction Count', 'refund_count':'Refund Count', 'txn_amt':'Transaction Amount (MYR)',
                                         'refund_amt':'Refund Amount (MYR)', 'refund_rate':'Refund Rate (%)', 'partial_refund_rate':'Partial Refund Rate (%)',
                                         'fully_refund_rate':'Fully Refund Rate (%)', 'refund_lvl':'Refund Level'})
display(asia_merged[:1])
asia_merged.to_csv(f'csv_file/Asia Merchants Refund Rate 2024.csv', index = False)

,Merchant ID,Country,Email,Phone Number,Channel,Transaction Count,Refund Count,Transaction Amount (MYR),Refund Amount (MYR),Refund Rate (%),Partial Refund Rate (%),Fully Refund Rate (%),Refund Level
12,Cox-Strong,Thailand,kathrynrobinson@gallegos.org,680-374-5677x459,"TNG eWallet, DuitNow QR Offline, FPX, Masterca...",1139,96,120153.21,6068.22,8.43,16.67,83.33,high


Checking

In [12]:
sqlInput = """
SELECT r.*
FROM refunds r
LEFT JOIN transactions t ON r.tran_id = t.tran_id
WHERE t.date BETWEEN '2025-01-01 00:00:00' AND '2025-12-31 23:59:59'
ORDER BY t.date desc
"""

export = pd.read_sql_query(sqlInput, conn)

export.to_csv(f'csv_file/2025 refunds.csv', index = False)

display(export)

,refund_id,tran_id,channel,amount,create_date,refund_date,reason,status
0,36f0a19a-2c3e-4adf-9bb4-26f71815c640,2494b5c1-c6c6-42b2-b731-60bfc908be89,Apple Pay,31.75,2025-12-31 23:43:17,None,Product Defect,pending
1,b49f4da4-3736-474a-b8ff-128506dc8fd8,629c9e63-d0b4-41f7-b23b-a6c6bee9865e,Apple Pay,19.62,2025-12-31 18:11:16,None,Wrong Item Sent,pending
2,d2080f5e-436d-42e9-af43-ce22362c4d2f,e307a36c-b1d5-4ef2-8132-fae7b91db00a,TNG eWallet,4.01,2025-12-31 17:27:18,None,Delayed Delivery,pending
3,92fe4995-77d5-474d-b867-a29dd29e21d8,45a22e10-4ca6-4a19-b5bc-c93d95f88e15,FPX,69.46,2025-12-31 16:18:47,None,Wrong Item Sent,pending
4,e66607cc-40f8-44eb-8bea-9da29c21a77a,5aa979e6-a344-42dc-8e7e-03a08d3180b3,DuitNow QR Offline,24.07,2025-12-31 11:59:15,2025-12-31 11:59:15,Fraudulent Transaction,completed
...,...,...,...,...,...,...,...,...
10087,5615c027-0fff-48d7-b1fe-643b98ca63c5,c54d7b97-94cd-4f32-a23e-0b1bb0ce58c4,DuitNow QR,102.57,2025-01-01 03:18:16,2025-01-01 03:18:16,Customer Dissatisfaction,completed
10088,b836f1d8-b34f-41de-891e-b3c10d796dab,7263160b-d5a1-4ad6-8038-48f4280c4a55,DuitNow QR Offline,20.62,2025-01-01 01:51:44,2025-01-01 01:51:44,Customer Dissatisfaction,completed
10089,1977c6ce-6a0c-407d-a313-8da13d4ae72d,b4399bf9-f7c5-4d86-b534-270ceac2dda0,Google Pay,28.22,2025-01-01 01:23:02,2025-01-01 01:23:02,Wrong Item Sent,completed
10090,62ae9451-5709-4350-ad9d-68f4837612ad,e9e3bf84-0fd9-4381-a811-feb8cfbf1f23,TNG eWallet Offline,34.43,2025-01-01 01:21:50,2025-01-01 01:21:50,Customer Dissatisfaction,completed


In [ ]:
sqlInput = """
SELECT date, merchant_id, tran_id, amount, currency
FROM transactions
WHERE date BETWEEN '2024-01-01 00:00:00' AND '2024-12-31 23:59:59'
AND status IN ('completed', 'recorded')
ORDER BY date desc
"""

txncheck = pd.read_sql_query(sqlInput, conn)

txncheck = convert_currency_to_MYR(txncheck)

txncheck = txncheck.groupby(["merchant_id"]).agg(
    counts=("tran_id", pd.Series.nunique),
    amount=("amount_MYR", 'sum')
).reset_index()

txncheck = txncheck.sort_values(by=['counts'], ascending=[False] )

display(txncheck)

,refund_id,tran_id,amount,date,reason,status
0,4829bfb4-9bcf-4ea8-8953-fb053addcd94,16f4f875-0d76-4c70-ba56-769615d0c042,126.04,2025-11-01 14:30:14,Product Defect,completed
1,eb963486-05f0-4b3f-835d-1b8d0ca3951f,4d459c37-2623-4f42-be69-8644ecaca32a,27.72,2025-01-28 17:43:33,Delayed Delivery,completed
2,0d1618f4-ec97-4a43-8e6c-cc477a2f8b8f,b3affb41-9c3a-4bec-9f7c-70f7cf56dafc,33.15,2025-07-29 23:48:40,Product Defect,completed
3,ebdb9954-2d73-4070-a4db-0d698cd409e1,14bb4c04-a9cd-4bfb-b695-6c2e52d02d9e,20.94,2025-04-24 09:52:29,Delayed Delivery,completed
4,ccce53f0-8942-4af7-acd7-63c5069cc54e,15b61faa-a0e9-40b4-97c6-f4e88a929bcf,31.69,2025-09-06 17:38:08,Other,completed
5,2e4cecd4-3e38-494d-8631-ba88472a9f7e,406e0943-fd22-4325-888f-86f434bf70ed,184.02,2025-10-11 03:37:10,Delayed Delivery,completed
6,cdfa68e6-cf5c-434a-8f52-35ab2ede2bae,47ac5831-7033-4cc8-b157-4fa2a350aec0,4.21,2025-10-10 06:35:12,Wrong Item Sent,completed
7,5dc5983f-c175-49ad-b89f-74d434d83fb2,668aa379-ca5c-416e-be72-d7bf7dd2e94e,18.27,2025-08-14 09:39:14,Delayed Delivery,cancelled
8,cc39109e-7262-459f-9406-5a993c2a07b5,b3fa2974-0356-475f-ae8e-db56c544f481,55.65,2025-02-19 16:33:21,Customer Dissatisfaction,completed
9,f15eec33-d9fe-490e-b987-3e1abdff19c1,cf1e2845-3aeb-49ae-8106-e03703ee84f4,25.41,2025-06-07 05:18:50,Delayed Delivery,cancelled


In [74]:
sqlInput = """
SELECT t.merchant_id, 
        COUNT(r.refund_id) AS refund_count
    FROM refunds r
    LEFT JOIN transactions t ON r.tran_id = t.tran_id
    WHERE t.date BETWEEN '2025-12-25 00:00:00' AND '2025-12-31 23:59:59' 
        AND r.status = 'completed' 
    GROUP BY t.merchant_id
"""

rrcheck = pd.read_sql_query(sqlInput, conn)

display(rrcheck)

,merchant_id,refund_count
0,Campbell Ltd,1
1,Gomez LLC,1
2,Hall and Sons,1
3,Johnson-Simpson,1
4,Morgan-Mcbride,1
5,"Rice, Prince and Johnson",1
6,Stein-Kelly,1
7,"Townsend, Bishop and Mccoy",33
8,Williams PLC,1


In [87]:
sqlInput = """
WITH refund_sum AS (
    SELECT t.merchant_id, 
        COUNT(r.refund_id) AS refund_count
    FROM refunds r
    LEFT JOIN transactions t ON r.tran_id = t.tran_id
    WHERE t.date BETWEEN '2025-12-25 00:00:00' AND '2025-12-31 23:59:59' 
        AND r.status = 'completed' 
    GROUP BY t.merchant_id
)
SELECT 
    t.merchant_id, 
    COUNT(t.tran_id) AS txn_count,
    COALESCE(r.refund_count, 0) AS refund_count,
    ROUND(
        COALESCE(r.refund_count, 0) * 100.0 
        / COUNT(t.tran_id), 
        2
    ) AS refund_rate
FROM transactions t
LEFT JOIN refund_sum r  ON t.merchant_id = r.merchant_id
WHERE t.status IN ('recorded', 'completed') 
    AND t.date BETWEEN '2025-12-25 00:00:00' AND '2025-12-31 23:59:59'
GROUP BY t.merchant_id
ORDER BY r.refund_count DESC
"""

rrcheck = pd.read_sql_query(sqlInput, conn)

display(rrcheck)

,merchant_id,txn_count,refund_count,refund_rate
0,"Townsend, Bishop and Mccoy",23,33,143.48
1,Williams PLC,13,1,7.69
2,Stein-Kelly,18,1,5.56
3,"Rice, Prince and Johnson",20,1,5.00
4,Morgan-Mcbride,10,1,10.00
...,...,...,...,...
108,Ashley-Thomas,7,0,0.00
109,Aguirre-Adams,7,0,0.00
110,"Aguilar, Roberts and Pruitt",15,0,0.00
111,Aguilar PLC,11,0,0.00
